# IC3Net 残差3层 demo —— 加载本地模型 + 推理打分

本 notebook 只负责**推理**: 加载 `ic3net_res_up_train.py` 产出的 `ic3net_res_up_model.json`,
在平台注入的**测试集区间**上打分, **不训练**。所有定义 (模型结构 / 特征工程 / cascade 预测 / load_models)
均在本 notebook 内自包含, 不依赖 train.py 导入。

架构: 3层残差叠加 f = f0 + Delta1+ + Delta2+
  分层: 0-33-67-100 (3组等分, label quantile)
  预测: 逐层剔除 cascading

流程拆成两个阶段:
1. **阶段一 (参赛者本地运行一次)** —— 执行 `python ic3net_res_up_train.py`
   在写死的训练区间 (2019-2020) 上从零训练, 把 **3模型权重 + 3组标准化统计 + 结构超参**
   存到 `ic3net_res_up_model.json`。
2. **阶段二 (平台公榜阶段调用 `main`)** —— 本 notebook 直接加载 JSON 权重, 在平台注入的
   测试区间上推理打分。

> 提交时请把 `ic3net_res_up_train.py` 与训练好的 `ic3net_res_up_model.json` 随本 notebook 一并上传。

In [ ]:
# ==== 自包含推理 notebook (不从 train.py 导入) ====
import os
import json
import time
import numpy as np
import pandas as pd
import dai
import structlog
import torch
import torch.nn as nn
import torch.nn.functional as F

logger = structlog.get_logger()

# ---------- 路径 ----------
if '__file__' in dir():
    _NB_DIR = os.path.dirname(os.path.abspath(__file__))
else:
    _NB_DIR = os.getcwd()
MODEL_PATH = os.path.join(_NB_DIR, "ic3net_res_up_model.json")

# ---------- 配置 (与 train.py 完全一致) ----------
N_GROUPS = 3
BATCH_PRED = 50000

FEATURE_COLS = [
    'open', 'high', 'low', 'close', 'volume', 'amount',
    'vwap', 'twap', 'vwap_twap_spread',
    'intraday_return', 'amplitude', 'close_position', 'real_body_ratio',
    'close_std', 'close_cv',
    'vol_concentration', 'vol_cv',
    'up_ratio', 'up_vol_ratio',
    'morning_vol_pct', 'afternoon_vol_pct',
]


# ---------- 轻量 Scaler ----------
class SimpleScaler:
    def __init__(self, mean, scale):
        self.mean = np.asarray(mean, dtype=np.float32)
        self.scale = np.asarray(scale, dtype=np.float32)
    def transform(self, X):
        return (X - self.mean) / self.scale


# ---------- 模型 (与 train.py 结构完全一致) ----------
class FactorMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=None):
        super().__init__()
        if hidden_dims is None: hidden_dims = [64, 32, 16]
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.Dropout(0.1))
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x).squeeze(-1)


class ResidualHead(nn.Module):
    def __init__(self, input_dim, hidden_dims=None):
        super().__init__()
        if hidden_dims is None: hidden_dims = [48, 24, 12]
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.Dropout(0.1))
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x).squeeze(-1)


# ---------- 特征工程 (与 train.py 完全一致) ----------
def build_features(bar1m_table, sd, ed):
    t0 = time.time()
    logger.info("build_features 开始", start=str(sd), end=str(ed))
    price_start = pd.to_datetime(sd) - pd.Timedelta(days=7)
    price_sql = f"""
    SELECT date_trunc('day', date)::DATE AS trading_day, instrument,
        ARG_MIN(open, date) AS open, MAX(high) AS high, MIN(low) AS low,
        ARG_MAX(close, date) AS close, SUM(volume) AS volume, SUM(amount) AS amount,
        SUM(amount)/NULLIF(SUM(volume),0) AS vwap, AVG(close) AS twap,
        (SUM(amount)/NULLIF(SUM(volume),0)-AVG(close))/NULLIF(ABS(AVG(close)),0) AS vwap_twap_spread,
        (ARG_MAX(close,date)-ARG_MIN(open,date))/NULLIF(ARG_MIN(open,date),0) AS intraday_return,
        (MAX(high)-MIN(low))/NULLIF(ARG_MIN(open,date),0) AS amplitude,
        (ARG_MAX(close,date)-MIN(low))/NULLIF(MAX(high)-MIN(low),0) AS close_position,
        ABS(ARG_MAX(close,date)-ARG_MIN(open,date))/NULLIF(MAX(high)-MIN(low),0) AS real_body_ratio,
        STDDEV(close) AS close_std, STDDEV(close)/NULLIF(ABS(AVG(close)),0) AS close_cv,
        CAST(MAX(volume) AS DOUBLE)/NULLIF(SUM(volume),0) AS vol_concentration,
        STDDEV(volume)/NULLIF(AVG(volume),0) AS vol_cv,
        CAST(SUM(CASE WHEN close>=open THEN 1 ELSE 0 END) AS DOUBLE)/NULLIF(COUNT(*),0) AS up_ratio,
        CAST(SUM(CASE WHEN close>=open THEN volume ELSE 0 END) AS DOUBLE)/NULLIF(SUM(volume),0) AS up_vol_ratio,
        CAST(SUM(CASE WHEN EXTRACT(HOUR FROM date)<12 THEN volume ELSE 0 END) AS DOUBLE)/NULLIF(SUM(volume),0) AS morning_vol_pct,
        CAST(SUM(CASE WHEN EXTRACT(HOUR FROM date)>=12 THEN volume ELSE 0 END) AS DOUBLE)/NULLIF(SUM(volume),0) AS afternoon_vol_pct
    FROM {bar1m_table} GROUP BY trading_day, instrument ORDER BY trading_day, instrument"""
    price = dai.query(price_sql, filters={'date': [price_start, ed]}).df().rename(
        columns={'trading_day': 'date'})
    price['date'] = pd.to_datetime(price['date'])
    price = price.sort_values(['instrument','date']).reset_index(drop=True)
    logger.info("量价特征完成", rows=len(price), elapsed=round(time.time()-t0,2))
    g = price.groupby('instrument', group_keys=False)['close']
    price['label'] = g.shift(-1) / price['close'] - 1
    df = price.copy()
    for col in FEATURE_COLS:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].replace([np.inf, -np.inf], np.nan)
    df = df[(df['date']>=pd.to_datetime(sd))&(df['date']<=pd.to_datetime(ed))]
    logger.info("build_features 结束", rows=len(df), total_elapsed=round(time.time()-t0,2))
    return df.reset_index(drop=True)


# ---------- 模型加载 (与 train.py save_models 配对的 load) ----------
def load_models(model_path=MODEL_PATH, map_location="cpu"):
    with open(model_path, "r", encoding="utf-8") as f:
        p = json.load(f)
    def _deserialize(tensors):
        sd = {}
        for k, meta in tensors.items():
            t = torch.tensor(meta["data"], dtype=getattr(torch, meta["dtype"]))
            sd[k] = t.reshape(meta["shape"]).to(map_location)
        return sd
    models = []
    models.append(FactorMLP(**p["model_cfg"]).to(map_location))
    models[0].load_state_dict(_deserialize(p["models"]["net0"]))
    models[0].eval()
    for i in range(1, p["n_groups"]):
        m = ResidualHead(**p["residual_cfg"]).to(map_location)
        m.load_state_dict(_deserialize(p["models"][f"net{i}"]))
        m.eval()
        models.append(m)
    scalers = [SimpleScaler(p["scalers"][str(i)]["mean"],
                            p["scalers"][str(i)]["scale"])
               for i in range(p["n_groups"])]
    return models, scalers, p["feature_cols"], p["n_groups"]


# ---------- Cascading 预测 ----------
def run_cascade_prediction(models, scalers, test_df, test_raw, device,
                           n_groups=N_GROUPS, batch_pred=BATCH_PRED):
    """逐层剔除 cascading 预测, 返回含 factor 列的 test_df"""
    t0 = time.time()
    for m in models: m.eval()
    N_test = len(test_raw)
    chains = np.zeros((N_test, n_groups), dtype=np.float32)

    for start in range(0, N_test, batch_pred):
        end = min(start + batch_pred, N_test)
        batch_raw = test_raw[start:end]
        x0 = torch.tensor(scalers[0].transform(batch_raw), dtype=torch.float32).to(device)
        with torch.no_grad():
            accum = models[0](x0)
        chains[start:end, 0] = accum.cpu().numpy()
        for k in range(1, n_groups):
            xk = torch.tensor(scalers[k].transform(batch_raw), dtype=torch.float32).to(device)
            with torch.no_grad():
                accum = accum + F.softplus(models[k](xk))
            chains[start:end, k] = accum.cpu().numpy()

    test_df['factor'] = 0.0
    n_covered = [0] * n_groups

    def _cascade(group):
        n = len(group)
        gi = group.index.values
        remaining = np.ones(n, dtype=bool)
        for g in range(n_groups - 1):
            n_rem = remaining.sum()
            if n_rem == 0: break
            pool_local = np.where(remaining)[0]
            pool_global = gi[pool_local]
            score_g = chains[pool_global, g]
            order = np.argsort(score_g)
            frac = 1.0 / (n_groups - g)
            n_take = max(1, int(n_rem * frac))
            take_local = pool_local[order[:n_take]]
            take_global = gi[take_local]
            group.loc[group.index[take_local], 'factor'] = chains[take_global, g]
            n_covered[g] += n_take
            remaining[take_local] = False
        if remaining.sum() > 0:
            last_local = np.where(remaining)[0]
            last_global = gi[last_local]
            g_last = n_groups - 1
            group.loc[group.index[last_local], 'factor'] = chains[last_global, g_last]
            n_covered[g_last] += len(last_local)
        return group

    test_df = test_df.groupby('date', group_keys=False).apply(_cascade)
    logger.info("cascade预测完成", total=len(test_df), elapsed=round(time.time()-t0,2),
                **{f'Q{i}': n_covered[i] for i in range(n_groups)})
    return test_df


# ==== 主入口 (平台公榜阶段调用) ====
def main(datasources, start_date, end_date):
    """加载已训练好的3层残差模型, 在样本外测试区间上 cascading 推理打分。

    start_date~end_date 为平台注入的【测试集区间】, 输出 ['date','instrument','factor']。"""
    table = datasources["bar1m"]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(
            f"未找到模型文件 {MODEL_PATH}; 请先运行 ic3net_res_up_train.py 训练并保存, 再随 notebook 一起上传")

    # 加载3模型 + 3scaler
    models, scalers, feature_cols, n_groups = load_models(MODEL_PATH, map_location=device)
    logger.info("已加载模型", path=MODEL_PATH, device=str(device),
                n_models=len(models), feature_cols=feature_cols)

    # 推理 (样本外测试区间, cascading)
    logger.info("构建测试集并预测", start=str(start_date), end=str(end_date))
    test_df = build_features(table, start_date, end_date)
    test_raw = test_df[feature_cols].values.astype(np.float64)
    test_raw = np.nan_to_num(test_raw, nan=0.0, posinf=0.0, neginf=0.0)

    test_df = run_cascade_prediction(models, scalers, test_df, test_raw, device, n_groups)

    # 对齐中证 1000
    stk_pool = dai.query("SELECT date, instrument FROM bigalpha_2026_instruments",
                         filters={'date': [start_date, end_date]}).df()
    stk_pool['instrument'] = stk_pool['instrument'].astype(str)

    result = pd.merge(test_df[['date', 'instrument', 'factor']], stk_pool,
                      how='inner', on=['date', 'instrument'])
    result['factor'] = result['factor'].replace([np.inf, -np.inf], np.nan)
    result = result.dropna(subset=['factor']).reset_index(drop=True)[
        ['date', 'instrument', 'factor']]
    logger.info("分数构建完成", rows=len(result), days=result["date"].nunique(),
                instruments=result["instrument"].nunique())
    return result


if __name__ == "__main__":
    from bigmodule import M

    datasources = {"bar1m": "bigalpha_2026_stock_bar1m"}

    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(
            f"未找到 {MODEL_PATH}, 请先运行 python ic3net_res_up_train.py 训练并生成权重文件")

    start_date, end_date = "2021-01-01 00:00:00", "2024-12-31 23:59:59"
    logger.info("计算分数 (仅加载权重推理, 不重训)", start=start_date, end=end_date)
    score_data = main(datasources, start_date, end_date)
    print(score_data.head())

    logger.info("开始评估分数")
    fp = dai.query("SELECT * FROM bigalpha_2026_factorlib",
                   filters={'date': [start_date, end_date]}).df()
    result = M.bigalpha_eval._latest(factor_data=score_data, factor_pool=fp,
                                     process_pools=False, show=True)
    logger.info("评估完成")